In [ ]:
!pip install cassandra-sigv4

In [18]:
from cassandra.cluster import Cluster
from ssl import SSLContext, PROTOCOL_TLSv1_2 , CERT_REQUIRED
from cassandra_sigv4.auth import SigV4AuthProvider
import boto3

ssl_context = SSLContext(PROTOCOL_TLSv1_2)
ssl_context.load_verify_locations('keyspaces-bundle.pem')
ssl_context.verify_mode = CERT_REQUIRED

/var/folders/rj/_dxpq5295j18zm80gp75719m0000gn/T/ipykernel_21723/1239316121.py:6: DeprecationWarning: ssl.PROTOCOL_TLSv1_2 is deprecated
  ssl_context = SSLContext(PROTOCOL_TLSv1_2)


In [21]:
# use this if you want to use Boto to set the session parameters.
auth_provider = SigV4AuthProvider(boto_session)

cluster = Cluster(['cassandra.us-east-1.amazonaws.com'], 
                  ssl_context=ssl_context, 
                  auth_provider=auth_provider,
                  port=9142)
session = cluster.connect()
r = session.execute('select * from system_schema.keyspaces')
print(r.current_rows)

[Row(keyspace_name='system_schema', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='system_schema_mcs', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='system', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='system_multiregion_info', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='de300_acharya', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='de300_barnett', durable_writes=True, replication=OrderedMa

### IN CLASS EXERCISE 

In [22]:
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS de300_rhemap
    WITH replication = {'class': 'SingleRegionStrategy'};
""")
print("Keyspace created!")

Keyspace created!


In [23]:
from cassandra import ConsistencyLevel
session.default_consistency_level = ConsistencyLevel.LOCAL_QUORUM

session.execute("""
    CREATE TABLE IF NOT EXISTS de300_rhemap.icu_stay_analysis (
        gender      TEXT,
        ethnicity   TEXT,
        subject_id  INT,
        icustay_id  INT,
        hadm_id     INT,
        los         DOUBLE,
        PRIMARY KEY ((gender, ethnicity), subject_id, icustay_id)
    );
""")
print("Table created!")

Table created!


/var/folders/rj/_dxpq5295j18zm80gp75719m0000gn/T/ipykernel_21723/1731621063.py:2: DeprecationWarning: Setting the consistency level at the session level will be removed in 4.0. Consider using execution profiles and setting the desired consistency level to the EXEC_PROFILE_DEFAULT profile.
  session.default_consistency_level = ConsistencyLevel.LOCAL_QUORUM


In [24]:
r = session.execute("SELECT * FROM system_schema.tables WHERE keyspace_name = 'de300_rhemap'")
print(r.current_rows)

[Row(keyspace_name='de300_rhemap', table_name='icu_stay_analysis', bloom_filter_fp_chance=0.01, caching=OrderedMapSerializedKey([('class', 'com.amazonaws.cassandra.DefaultCaching')]), cdc=False, comment='', compaction=OrderedMapSerializedKey([('class', 'com.amazonaws.cassandra.DefaultCompaction')]), compression=OrderedMapSerializedKey([('class', 'com.amazonaws.cassandra.DefaultCompression')]), crc_check_chance=1.0, dclocal_read_repair_chance=0.0, default_time_to_live=0, extensions=OrderedMapSerializedKey([]), flags=SortedSet(['compound']), gc_grace_seconds=7776000, id=UUID('e7199dac-0138-3380-80b4-00a0e05eeee0'), max_index_interval=2048, memtable_flush_period_in_ms=3600000, min_index_interval=128, read_repair_chance=0.0, speculative_retry='99PERCENTILE')]


In [32]:
icustays   = pd.read_csv('ICUSTAYS.csv')
patients   = pd.read_csv('PATIENTS.csv')
admissions = pd.read_csv('ADMISSIONS.csv')

print(icustays.shape)
print(patients.shape)
print(admissions.shape)

(136, 12)
(100, 8)
(129, 19)


In [33]:
df = icustays.merge(patients[['subject_id', 'gender']], on='subject_id')

df = df.merge(admissions[['hadm_id', 'ethnicity']], on='hadm_id')

df = df[['gender', 'ethnicity', 'subject_id', 'icustay_id', 'hadm_id', 'los']]

df = df.dropna()


In [35]:
from cassandra.query import SimpleStatement

insert_query = session.prepare("""
    INSERT INTO de300_rhemap.icu_stay_analysis 
    (gender, ethnicity, subject_id, icustay_id, hadm_id, los)
    VALUES (?, ?, ?, ?, ?, ?)
""")

# Loop through rows and insert
for _, row in df.iterrows():
    session.execute(insert_query, (
        row['gender'],
        row['ethnicity'],
        int(row['subject_id']),
        int(row['icustay_id']),
        int(row['hadm_id']),
        float(row['los'])
    ))

print(f"Inserted {len(df)} rows!")

Inserted 136 rows!


In [37]:
session.shutdown()